In [1]:
# Data Cleaning Notebook
# This notebook cleans and preprocesses the three datasets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# Create processed data directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Set plotting style
plt.style.use('ggplot')
sns.set(font_scale=1.2)

print("NYC Data Analysis - Data Cleaning")
print("=" * 50)

NYC Data Analysis - Data Cleaning


In [3]:
# 1. Clean Crime Data
print("\n1. Cleaning Crime Data\n" + "-" * 30)

# Load the Excel file
try:
    # Try reading with specified sheet name
    crime_data = pd.read_excel('../data/311_Crime_NYPD.xlsx', sheet_name='311_Service_Requests_from_2010_')
    print("Successfully loaded crime data with specified sheet name")
except:
    # If that fails, try reading the second sheet (index 1)
    try:
        crime_data = pd.read_excel('../data/311_Crime_NYPD.xlsx', sheet_name=1)
        print("Successfully loaded crime data from second sheet")
    except:
        # If all fails, just read the first available sheet
        crime_data = pd.read_excel('../data/311_Crime_NYPD.xlsx')
        print("Loaded crime data from default sheet")


1. Cleaning Crime Data
------------------------------
Successfully loaded crime data with specified sheet name


In [4]:
# Display basic information
print(f"Crime data shape before cleaning: {crime_data.shape}")

# Check column names
print("Original column names:")
print(crime_data.columns.tolist())

Crime data shape before cleaning: (92765, 5)
Original column names:
['Unique Key', 'by_year_created_date', 'Incident Zip', 'Descriptor', 'Year']


In [17]:
# Standardize column names based on what we observed in exploration
# Rename columns if needed
if 'Sum of Unique Key' in crime_data.columns:
    # This is the format we observed in the exploration
    crime_data_clean = crime_data.copy()
    # Rename columns
    if crime_data_clean.columns[1] is None:
        crime_data_clean.columns = ['metric', 'zip_code' if pd.isna(crime_data_clean.columns[1]) else crime_data_clean.columns[1]] + list(crime_data_clean.columns[2:])
    # Melt the data to convert wide format to long format
    id_vars = ['Year', 'Incident Zip'] if 'Year' in crime_data_clean.columns and 'Incident Zip' in crime_data_clean.columns else ['Year', crime_data_clean.columns[1]]
    crime_data_clean = pd.melt(crime_data_clean, 
                              id_vars=id_vars, 
                              var_name='crime_type', 
                              value_name='count')
    print("\nConverted crime data to long format")
else:
    # If the format is different, do basic cleaning
    crime_data_clean = crime_data.copy()
    # Rename columns to lowercase with underscores
    crime_data_clean.columns = [str(col).lower().replace(' ', '_') for col in crime_data_clean.columns]
    print("\nRenamed columns to lowercase with underscores")


Renamed columns to lowercase with underscores


In [18]:
crime_data_clean = crime_data_clean.dropna()

In [19]:
# Make sure year and zip are proper types
try:
    #year_col = [col for col in crime_data_clean.columns if 'year' in col.lower()][0]
    crime_data_clean['year'] = pd.to_numeric(crime_data_clean['year'], errors='coerce')
    crime_data_clean = crime_data_clean.dropna(subset=['year'])
    crime_data_clean['year'] = crime_data_clean['year'].astype(int)
    
    zip_col = [col for col in crime_data_clean.columns if 'zip' in col.lower()][0]
    crime_data_clean[zip_col] = pd.to_numeric(crime_data_clean[zip_col], errors='coerce')
    crime_data_clean = crime_data_clean.dropna(subset=[zip_col])
    crime_data_clean[zip_col] = crime_data_clean[zip_col].astype(int)
    
    # Standardize column names
    crime_data_clean = crime_data_clean.rename(columns={zip_col: 'zip_code'})
    
except Exception as e:
    print(f"Error standardizing year and zip columns: {e}")

In [20]:
# Display after cleaning
print(f"\nCrime data shape after cleaning: {crime_data_clean.shape}")
print("\nCleaned crime data sample:")
print(crime_data_clean.head())



Crime data shape after cleaning: (89749, 5)

Cleaned crime data sample:
   unique_key by_year_created_date  zip_code                     descriptor  \
0          10           2011-01-01     10014                     Unlicensed   
1          57           2011-01-01     11231  Posted Parking Sign Violation   
2          26           2011-01-01     11358                   Loud Talking   
3         305           2011-01-01     11233                      No Access   
4          35           2011-01-01     10011                Blocked Hydrant   

   year  
0  2011  
1  2011  
2  2011  
3  2011  
4  2011  


In [40]:
crime_data = crime_data_clean
# Find the ZIP code, year, and descriptor columns
zip_col = [col for col in crime_data.columns if 'zip' in col.lower()]
year_col = [col for col in crime_data.columns if col.lower() == 'year']
descriptor_col = [col for col in crime_data.columns if 'descriptor' in col.lower()]

if not zip_col or not year_col or not descriptor_col:
    print("Warning: Could not identify key columns. Please check column names.")
    print("Available columns:", crime_data.columns.tolist())
    exit(1)

# Use the first matching column name
zip_col = zip_col[0]
year_col = year_col[0]
descriptor_col = descriptor_col[0]

print(f"Using columns: {zip_col}, {year_col}, {descriptor_col}")

# Keep only the columns we need
crime_data = crime_data[[zip_col, year_col, descriptor_col]].copy()

# Convert data types
crime_data[year_col] = pd.to_numeric(crime_data[year_col], errors='coerce')
crime_data[zip_col] = pd.to_numeric(crime_data[zip_col], errors='coerce')

Using columns: zip_code, year, descriptor


In [41]:
# Drop rows with missing values
crime_data = crime_data.dropna()

# Rename columns for consistency
crime_data = crime_data.rename(columns={
    zip_col: 'zip_code',
    year_col: 'year',
    descriptor_col: 'crime_type'
})

print(f"Cleaned data shape: {crime_data.shape}")

# Aggregate by zip_code, year, and crime_type
print("Aggregating crime data...")
crime_counts = crime_data.groupby(['zip_code', 'year', 'crime_type']).size().reset_index(name='count')
print(f"Aggregated data shape: {crime_counts.shape}")

# Also create total counts by zip_code and year
total_counts = crime_counts.groupby(['zip_code', 'year'])['count'].sum().reset_index(name='total_crime_count')
print(f"Total counts shape: {total_counts.shape}")

Cleaned data shape: (89749, 3)
Aggregating crime data...
Aggregated data shape: (89749, 4)
Total counts shape: (3033, 3)


In [42]:
# Save the aggregated data
crime_counts.to_csv('../data/processed/crime_data_cleaned.csv', index=False)
total_counts.to_csv('../data/processed/crime_data_total.csv', index=False)

print("Data saved to '../data/processed/crime_data_cleaned.csv' and '../data/processed/crime_data_total.csv'")

# Print some sample data
print("\nSample of aggregated crime data:")
print(crime_counts.head(10))

print("\nSample of total crime counts:")
print(total_counts.head(10))

Data saved to '../data/processed/crime_data_cleaned.csv' and '../data/processed/crime_data_total.csv'

Sample of aggregated crime data:
   zip_code  year                     crime_type  count
0        83  2013               Loud Music/Party      1
1        83  2013                   Loud Talking      1
2        83  2013                Loud Television      1
3        83  2014               Loud Music/Party      1
4        83  2014                   Loud Talking      1
5        83  2014  Posted Parking Sign Violation      1
6        83  2014       Unauthorized Bus Layover      1
7        83  2015               Loud Music/Party      1
8        83  2015  Posted Parking Sign Violation      1
9        83  2015       Unauthorized Bus Layover      1

Sample of total crime counts:
   zip_code  year  total_crime_count
0        83  2013                  3
1        83  2014                  4
2        83  2015                  3
3        83  2016                  5
4        83  2017               

In [43]:
# Now create the one-hot encoded version
print("Creating one-hot encoded version...")

# Get top 50 most common crime types to avoid too many columns
top_crimes = crime_counts.groupby('crime_type')['count'].sum().sort_values(ascending=False).head(50).index.tolist()
print(f"Selected top {len(top_crimes)} crime types for one-hot encoding")

# Filter to only include top crime types
crime_counts_top = crime_counts[crime_counts['crime_type'].isin(top_crimes)]

# Create a pivot table (one-hot encoding)
crime_onehot = crime_counts_top.pivot_table(
    index=['zip_code', 'year'],
    columns='crime_type',
    values='count',
    aggfunc='sum',
    fill_value=0
).reset_index()

print(f"One-hot encoded data shape: {crime_onehot.shape}")

# Save the one-hot encoded version
crime_onehot.to_csv('../data/processed/crime_data_onehot.csv', index=False)
print("Saved one-hot encoded data to '../data/processed/crime_data_onehot.csv'")

Creating one-hot encoded version...
Selected top 50 crime types for one-hot encoding
One-hot encoded data shape: (3021, 52)
Saved one-hot encoded data to '../data/processed/crime_data_onehot.csv'


In [44]:
# Print some sample data
print("\nSample of one-hot encoded crime data:")
print(crime_onehot.head(5))


Sample of one-hot encoded crime data:
crime_type  zip_code  year  After Hours - Licensed Est  Banging/Pounding  \
0                 83  2013                           0                 0   
1                 83  2014                           0                 0   
2                 83  2015                           0                 0   
3                 83  2016                           0                 1   
4                 83  2017                           0                 0   

crime_type  Blocked Bike Lane  Blocked Crosswalk  Blocked Hydrant  \
0                           0                  0                0   
1                           0                  0                0   
2                           0                  0                0   
3                           0                  0                1   
4                           0                  0                0   

crime_type  Blocked Sidewalk  Building  Car/Truck Horn  ...  Tortured  \
0               

In [22]:
# 2. Clean ACS Demographics Data
print("\n\n2. Cleaning ACS Demographics Data\n" + "-" * 40)

# Load the CSV file
acs_data = pd.read_csv('../data/ACS_update.csv')

# Display basic information
print(f"ACS data shape before cleaning: {acs_data.shape}")

# Standardize column names to lowercase with underscores
acs_data_clean = acs_data.copy()



2. Cleaning ACS Demographics Data
----------------------------------------
ACS data shape before cleaning: (2348, 28)


In [23]:
# Check for missing values
missing_count = acs_data_clean.isnull().sum().sum()
print(f"\nMissing values before cleaning: {missing_count}")

# Handle missing values
# For numeric columns, fill with median
numeric_cols = acs_data_clean.select_dtypes(include=['number']).columns
for col in numeric_cols:
    acs_data_clean[col] = acs_data_clean[col].fillna(acs_data_clean[col].median())

# Check missing values after cleaning
missing_count_after = acs_data_clean.isnull().sum().sum()
print(f"Missing values after cleaning: {missing_count_after}")


Missing values before cleaning: 2699
Missing values after cleaning: 0


In [24]:
# Rename zcta to zip_code for consistency across datasets
acs_data_clean = acs_data_clean.rename(columns={'zcta': 'zip_code'})

# Display after cleaning
print(f"\nACS data shape after cleaning: {acs_data_clean.shape}")
print("\nCleaned ACS data sample:")
print(acs_data_clean.head())


ACS data shape after cleaning: (2348, 28)

Cleaned ACS data sample:
   zip_code  year   merge_key  number_of_households  \
0     10001  2023  2023_10001                 15097   
1     10002  2023  2023_10002                 35771   
2     10003  2023  2023_10003                 25080   
3     10004  2023  2023_10004                  1775   
4     10005  2023  2023_10005                  5156   

   household_median_income_1yr  household_mean_income_1yr  \
0                       123393                     205444   
1                        46525                      93314   
2                       153750                     248577   
3                       220592                     309645   
4                       211810                     279387   

   number_of_rent_home  median_rent  number_of_owned_home  median_home_value  \
0                11399         3024                  3570             793000   
1                28916         1207                  6660             808

In [25]:
# Save the cleaned data
acs_data_clean.to_csv('../data/processed/demographics_data_cleaned.csv', index=False)
print("\nSaved cleaned ACS data to '../data/processed/demographics_data_cleaned.csv'")


Saved cleaned ACS data to '../data/processed/demographics_data_cleaned.csv'


In [35]:
# 3. Clean Business and Gentrification Data
print("\n\n3. Cleaning Business and Gentrification Data\n" + "-" * 50)

# Load the CSV file
business_data = pd.read_csv('../data/business_type_gentrification.csv')

# Display basic information
print(f"Business data shape before cleaning: {business_data.shape}")

# Standardize column names to lowercase with underscores
business_data_clean = business_data.copy()



3. Cleaning Business and Gentrification Data
--------------------------------------------------
Business data shape before cleaning: (420, 134)


In [36]:
# Rename the first column assuming it's the ZIP code
business_data_clean = business_data_clean.rename(columns={business_data_clean.columns[0]: 'zip_code'})

# Rename yr to year for consistency
business_data_clean = business_data_clean.rename(columns={'yr': 'year'})

# Check for missing values
missing_count = business_data_clean.isnull().sum().sum()
print(f"\nMissing values before cleaning: {missing_count}")


Missing values before cleaning: 0


In [37]:
# Handle missing values
# For numeric columns, fill with median
numeric_cols = business_data_clean.select_dtypes(include=['number']).columns
for col in numeric_cols:
    business_data_clean[col] = business_data_clean[col].fillna(business_data_clean[col].median())

# Check missing values after cleaning
missing_count_after = business_data_clean.isnull().sum().sum()
print(f"Missing values after cleaning: {missing_count_after}")

Missing values after cleaning: 0


In [38]:
# Display after cleaning
print(f"\nBusiness data shape after cleaning: {business_data_clean.shape}")
print("\nCleaned business data sample (selected columns):")
# Show just a subset of columns since there are many
selected_cols = ['zip_code', 'year', 'is_gentrified', 'rent', 'rent_change', 'EXPENSIVE', 'INEXPENSIVE', 'MODERATE']
print(business_data_clean[selected_cols].head())


Business data shape after cleaning: (420, 134)

Cleaned business data sample (selected columns):
   zip_code  year  is_gentrified         rent  rent_change  EXPENSIVE  \
0     10001  2022          False  4870.677598     0.205286       17.0   
1     10002  2022           True  3554.796326     0.244991       12.0   
2     10003  2022           True  3941.807689     0.256503       25.0   
3     10004  2022          False  3946.666521     0.044439        2.0   
4     10005  2022          False  4275.182475     0.217989        4.0   

   INEXPENSIVE  MODERATE  
0         25.0      90.0  
1         62.0      75.0  
2         68.0     108.0  
3         13.0      11.0  
4          1.0       5.0  


In [39]:
# Save the cleaned data
business_data_clean.to_csv('../data/processed/business_data_cleaned.csv', index=False)
print("\nSaved cleaned business data to '../data/processed/business_data_cleaned.csv'")

print("\n\nAll datasets have been cleaned and saved to the processed directory.")



Saved cleaned business data to '../data/processed/business_data_cleaned.csv'


All datasets have been cleaned and saved to the processed directory.
